# Classical / centrality features — dataset_3 (ER, avg degree 0.8)

Same extraction as Dataset 1/2. The ER network is structure-only (no balance sheet), so **DebtRank is
skipped** (it needs equity); the other structural centralities are computed.

Output: `src/data/classical_features/dataset_3/classical_features_dataset3.parquet`

In [1]:
import sys
from pathlib import Path

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'src').exists() and (path / 'requirements.txt').exists():
            return path
    raise FileNotFoundError('Project root not found.')

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.models import (
    degree_centrality, betweenness_centrality, closeness_centrality,
    eigenvector_centrality, weighted_degree, pagerank_centrality,
)

DATASET_DIR  = PROJECT_ROOT / 'src' / 'datasets' / 'dataset_3'
OUT_FEATURES = PROJECT_ROOT / 'src' / 'data' / 'classical_features' / 'dataset_3'
OUT_FEATURES.mkdir(parents=True, exist_ok=True)
N = 10000  # ER network has 10000 nodes (ids 1..N), incl. isolated

## Load the network

In [2]:
e = pd.read_csv(DATASET_DIR / 'edges.csv')
edges = e.rename(columns={'from': 'Sourceid', 'to': 'Targetid', 'weight': 'Weights'})[['Sourceid', 'Targetid', 'Weights']]
nodes = pd.DataFrame({'index': np.arange(1, N + 1)})   # all nodes, incl. isolated -> 0 centrality
print('edges:', edges.shape, '| nodes:', nodes.shape)
edges.head(3)

edges: (9056, 3) | nodes: (10000, 1)


,Sourceid,Targetid,Weights
0,1,1199,0.066667
1,1,5199,0.050000
2,1,9820,0.100000


## Compute centralities (DebtRank skipped — no equity in the ER network)

In [3]:
deg_df  = degree_centrality(edges, nodes)
bet_df  = betweenness_centrality(edges, nodes)
clo_df  = closeness_centrality(edges, nodes)
wdeg_df = weighted_degree(edges, nodes)
pr_df   = pagerank_centrality(edges, nodes, reverse=True)

features = nodes.rename(columns={'index': 'bank_id'}).copy()
for df in (deg_df, bet_df, clo_df, wdeg_df, pr_df):
    features = features.merge(df, on='bank_id', how='left')

NETWORK_METRICS = [
    'degree_centrality_total',
    'weighted_degree_in', 'weighted_degree_out', 'weighted_degree_total',
    'betweenness_centrality', 'closeness_centrality', 'pagerank',
]
features = features[['bank_id'] + [c for c in NETWORK_METRICS if c in features.columns]]
out_path = OUT_FEATURES / 'classical_features_dataset3.parquet'
features.to_parquet(out_path, index=False)
print(f'[OK] saved {out_path.name}  shape={features.shape}')
print('columns:', features.columns.tolist())

[OK] saved classical_features_dataset3.parquet  shape=(10000, 8)
columns: ['bank_id', 'degree_centrality_total', 'weighted_degree_in', 'weighted_degree_out', 'weighted_degree_total', 'betweenness_centrality', 'closeness_centrality', 'pagerank']


## Add eigenvector centrality (computed separately; may fail to converge)

In [4]:
out_path = OUT_FEATURES / 'classical_features_dataset3.parquet'
features = pd.read_parquet(out_path)
try:
    eig_df = eigenvector_centrality(edges, nodes)
    features = features.merge(eig_df, on='bank_id', how='left')
    features.to_parquet(out_path, index=False)
    print(f'[OK] added eigenvector_centrality  shape={features.shape}')
except Exception as ex:
    print('eigenvector_centrality skipped (did not converge):', type(ex).__name__, ex)

[OK] added eigenvector_centrality  shape=(10000, 9)


## Verify

In [5]:
df = pd.read_parquet(OUT_FEATURES / 'classical_features_dataset3.parquet')
print(f'Shape: {df.shape}')
display(df.head())

feature_cols = [
    'degree_centrality_total', 'weighted_degree_total',
    'betweenness_centrality', 'closeness_centrality',
    'eigenvector_centrality', 'pagerank',
]
print('\nFeature summary:')
display(df[feature_cols].describe())

Shape: (10000, 9)


,bank_id,degree_centrality_total,weighted_degree_in,weighted_degree_out,weighted_degree_total,betweenness_centrality,closeness_centrality,pagerank,eigenvector_centrality
0,1,0.000357,0.0,0.216667,0.216667,0.0,0.000000,0.000108,2.835608e-25
1,2,0.000119,0.0,0.100000,0.100000,0.0,0.000000,0.000170,2.835608e-25
2,3,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000e+00
3,4,0.000119,0.2,0.000000,0.200000,0.0,0.000119,0.000054,3.482126e-23
4,5,0.000357,0.2,0.000000,0.200000,0.0,0.000590,0.000054,1.326154e-14



Feature summary:


,degree_centrality_total,weighted_degree_total,betweenness_centrality,closeness_centrality,eigenvector_centrality,pagerank
count,10000.000000,10000.000000,1.000000e+04,10000.000000,1.000000e+04,10000.000000
mean,0.000216,0.238760,8.719901e-07,0.000215,2.955804e-04,0.000100
std,0.000160,0.168235,4.892391e-06,0.000312,9.996130e-03,0.000086
min,0.000000,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000
25%,0.000119,0.100000,0.000000e+00,0.000000,2.835608e-25,0.000054
50%,0.000238,0.200000,0.000000e+00,0.000119,3.482126e-23,0.000076
75%,0.000357,0.335000,8.508465e-08,0.000292,8.711036e-20,0.000135
max,0.001072,1.100000,1.310445e-04,0.003059,6.797781e-01,0.000890
